### Recipe Audit STATION CSV 
Recipe Audit Station CSV adalah panduan atau prosedur sistematis untuk memeriksa kualitas data pengamatan yang disimpan dalam format CSV (Comma-Separated Values).

Data stasiun pengamatan dapat berupa data temperatur, tekanan, kelembapan, atau parameter lainnya yang dicatat berdasarkan waktu. Sebelum data tersebut digunakan untuk analisis, kita perlu memastikan bahwa strukturnya benar, nilainya tidak banyak yang hilang, dan pencatatan waktunya konsisten.

In [ ]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

# 1. Configuration
CSV_PATH = Path("station.csv")
TIME_COLUMN = "timestamp"
UNITS = {
    "temperature": "degC",
    "pressure": "hPa",
    "humidity": "%"
}

REPORT_PATH = Path("audit_report.md")
RUN_PATH = Path("audit_run.json")


def audit_csv(csv_path, time_column, units):
    # Read input without modifying the original file
    df = pd.read_csv(csv_path)

    report = []
    report.append("# Station CSV Audit Report")
    report.append(f"\nFile: {csv_path}")
    report.append(f"\nRows: {len(df)}")
    report.append(f"\nColumns: {list(df.columns)}")

    # 2. Schema checks
    required_columns = [time_column] + list(units.keys())
    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    report.append("\n## Schema Check")
    report.append(f"Missing columns: {missing_columns}")

    # 3. Missing values
    report.append("\n## Missing Values")
    report.append(df.isna().sum().to_string())

    # 4. Timestamp validation
    report.append("\n## Timestamp Check")

    if time_column in df.columns:
        times = pd.to_datetime(
            df[time_column], errors="coerce", utc=True
        )

        invalid_times = int(times.isna().sum())
        duplicate_times = int(
            times.dropna().duplicated().sum()
        )

        valid_times = times.dropna()

        report.append(f"Invalid timestamps: {invalid_times}")
        report.append(f"Duplicate timestamps: {duplicate_times}")

        if not valid_times.empty:
            report.append(f"Start: {valid_times.min()}")
            report.append(f"End: {valid_times.max()}")

    # 5. Unit contract
    report.append("\n## Unit Contract")
    for column, unit in units.items():
        report.append(f"- {column}: {unit}")

    # 6. Save reports
    REPORT_PATH.write_text(
        "\n".join(report), encoding="utf-8"
    )

    run_record = {
        "input": str(csv_path),
        "report": str(REPORT_PATH),
        "rows": len(df),
        "audit_generated_at": datetime.now().astimezone().isoformat(),
        "execution_status": "completed"
    }

    RUN_PATH.write_text(
        json.dumps(run_record, indent=2),
        encoding="utf-8"
    )

    print("Audit completed.")
    print(f"Report: {REPORT_PATH}")
    print(f"Run record: {RUN_PATH}")


if __name__ == "__main__":
    audit_csv(CSV_PATH, TIME_COLUMN, UNITS)

### Recipe Normalisasi GeoJSON
Recipe Normalisasi GeoJSON adalah prosedur untuk memeriksa dan menyeragamkan struktur data spasial dalam format GeoJSON tanpa menghilangkan informasi penting dari fitur geografis.

GeoJSON adalah format pertukaran data geografis berbasis JSON. Format ini dapat menyimpan objek spasial seperti titik lokasi stasiun, garis jalan, dan poligon batas wilayah.

In [ ]:
import json
import csv
from pathlib import Path
from collections import Counter

INPUT_PATH = Path("input.geojson")
OUTPUT_PATH = Path("normalized.geojson")
INVENTORY_PATH = Path("inventory.csv")
SUMMARY_PATH = Path("summary.json")

# Must be confirmed from the source metadata.
CRS_CONTRACT = "Not verified"
AXIS_CONTRACT = "Not verified"


def normalize_geojson(input_path):
    data = json.loads(
        input_path.read_text(encoding="utf-8")
    )

    if data.get("type") != "FeatureCollection":
        raise ValueError(
            "Expected a GeoJSON FeatureCollection."
        )

    features = data.get("features")
    if not isinstance(features, list):
        raise ValueError("'features' must be a list.")

    normalized_features = []
    inventory = []
    geometry_types = Counter()

    for index, feature in enumerate(features):
        if not isinstance(feature, dict):
            raise ValueError(f"Invalid feature at index {index}")

        if feature.get("type") != "Feature":
            raise ValueError(f"Feature type error at index {index}")

        properties = feature.get("properties")
        if properties is not None and not isinstance(properties, dict):
            raise ValueError(
                f"Invalid properties at index {index}"
            )

        geometry = feature.get("geometry")

        if geometry is not None:
            if not isinstance(geometry, dict):
                raise ValueError(
                    f"Invalid geometry at index {index}"
                )

            geometry_type = geometry.get("type", "Unknown")
            geometry_types[geometry_type] += 1
        else:
            geometry_type = "Null"

        # Preserve original feature content and coordinate order.
        normalized_feature = dict(feature)

        if "properties" not in normalized_feature:
            normalized_feature["properties"] = None

        if "geometry" not in normalized_feature:
            normalized_feature["geometry"] = None

        normalized_features.append(normalized_feature)

        inventory.append({
            "feature_index": index,
            "feature_id": feature.get("id"),
            "geometry_type": geometry_type,
            "null_geometry": geometry is None,
            "properties_is_null": properties is None,
            "property_count": (
                len(properties)
                if isinstance(properties, dict)
                else 0
            )
        })

    # Preserve top-level metadata and replace only features.
    normalized_data = dict(data)
    normalized_data["features"] = normalized_features

    OUTPUT_PATH.write_text(
        json.dumps(
            normalized_data,
            ensure_ascii=False,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    with INVENTORY_PATH.open(
        "w", newline="", encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
                "feature_index", "feature_id",
                "geometry_type", "null_geometry",
                "properties_is_null", "property_count"
            ]
        )
        writer.writeheader()
        writer.writerows(inventory)

    summary = {
        "input": str(input_path),
        "output": str(OUTPUT_PATH),
        "feature_count_input": len(features),
        "feature_count_output": len(normalized_features),
        "geometry_types": dict(geometry_types),
        "null_geometry_count": sum(
            item["null_geometry"] for item in inventory
        ),
        "null_properties_count": sum(
            item["properties_is_null"] for item in inventory
        ),
        "crs_contract": CRS_CONTRACT,
        "axis_contract": AXIS_CONTRACT,
        "normalization_status": "completed"
    }

    SUMMARY_PATH.write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8"
    )

    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    normalize_geojson(INPUT_PATH)

### Recipe Model Difusi Panas Satu Dimensi
Recipe Model Difusi Panas Satu Dimensi adalah prosedur untuk mensimulasikan perpindahan panas sepanjang satu arah, misalnya dari ujung kiri ke ujung kanan suatu batang.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# 1. Configuration
LENGTH = 1.0          # Rod length (m)
ALPHA = 1.0e-4        # Thermal diffusivity (m^2/s)
DZ = 0.02             # Spatial grid spacing (m)
DT = 1.0              # Time step (s)
TOTAL_TIME = 100.0    # Total simulation time (s)

LEFT_TEMP = 100.0     # Left boundary temperature (degC)
RIGHT_TEMP = 20.0     # Right boundary temperature (degC)
INITIAL_TEMP = 20.0   # Initial interior temperature (degC)

OUTPUT_DIR = Path("heat_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# 2. Build spatial and temporal grids
if LENGTH <= 0 or DZ <= 0 or DT <= 0:
    raise ValueError("Length, DZ, and DT must be positive.")

if ALPHA <= 0 or TOTAL_TIME < 0:
    raise ValueError("Invalid diffusivity or simulation time.")

n_intervals = round(LENGTH / DZ)

if not np.isclose(n_intervals * DZ, LENGTH):
    raise ValueError("DZ must divide LENGTH evenly.")

x = np.linspace(0, LENGTH, n_intervals + 1)
n_steps = round(TOTAL_TIME / DT)

if not np.isclose(n_steps * DT, TOTAL_TIME):
    raise ValueError("DT must divide TOTAL_TIME evenly.")

# 3. Stability check for explicit FTCS
r = ALPHA * DT / DZ**2

if r > 0.5:
    raise ValueError(
        f"Unstable configuration: r={r:.4f}. "
        "Reduce DT or increase DZ."
    )

# 4. Initial temperature profile
T = np.full(len(x), INITIAL_TEMP, dtype=float)
T[0] = LEFT_TEMP
T[-1] = RIGHT_TEMP

# Save selected profiles
profiles = [T.copy()]
profile_times = [0.0]

# 5. Time integration
for step in range(1, n_steps + 1):
    old_T = T.copy()

    # FTCS finite-difference update
    T[1:-1] = old_T[1:-1] + r * (
        old_T[2:] - 2 * old_T[1:-1] + old_T[:-2]
    )

    # Enforce fixed-temperature boundary conditions
    T[0] = LEFT_TEMP
    T[-1] = RIGHT_TEMP

    if step % max(1, n_steps // 5) == 0:
        profiles.append(T.copy())
        profile_times.append(step * DT)

# 6. Plot temperature profiles
for profile, time in zip(profiles, profile_times):
    plt.plot(x, profile, label=f"t = {time:.1f} s")

plt.xlabel("Position (m)")
plt.ylabel("Temperature (degC)")
plt.title("1D Heat Diffusion")
plt.legend()
plt.grid(True)

figure_path = OUTPUT_DIR / "temperature_profiles.png"
plt.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()

# 7. Save numerical output
np.savetxt(
    OUTPUT_DIR / "final_temperature.csv",
    np.column_stack((x, T)),
    delimiter=",",
    header="position_m,temperature_degC",
    comments=""
)

# 8. Save run record
run_record = {
    "length_m": LENGTH,
    "thermal_diffusivity_m2_s": ALPHA,
    "dz_m": DZ,
    "dt_s": DT,
    "total_time_s": TOTAL_TIME,
    "stability_ratio": r,
    "stability_limit": 0.5,
    "number_of_grid_points": len(x),
    "number_of_time_steps": n_steps,
    "boundary_conditions": {
        "left_degC": LEFT_TEMP,
        "right_degC": RIGHT_TEMP
    },
    "figure": str(figure_path),
    "execution_status": "completed"
}

(OUTPUT_DIR / "run_record.json").write_text(
    json.dumps(run_record, indent=2),
    encoding="utf-8"
)

print("Simulation completed.")
print(f"Stability ratio r = {r:.4f}")
print(f"Outputs saved to: {OUTPUT_DIR.resolve()}")